# 03 · Module 3: Fraud detection

**Question.** Which transactions to flag, and what that saves against today's practice.

The deliverable that matters is the cost curve.

**Data.** `nova_transactions.csv`: 30,000 transactions, Jan 2025 – Aug 2026, Fraud_Flag 2.3%. Split by time: first 70% train, next 15% validate, last 15% test (`splits.time_split`).

**Professor's anchor.** S9 (rules, ML scoring, alerts; Isolation Forest; the hybrid RF + IF score), TA6 (resample inside folds; contamination), S4 (the fraud threshold example).

In [ ]:
import sys
sys.path.insert(0, "../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from novakit import ASSUMPTIONS, CRORE, LAKH, SEED, load, use_module
from novakit import evaluate, explain, finance, handoff, splits

RES, FIG = use_module("03")

In [ ]:
tx = load("transactions", parse_dates=["Timestamp"]).sort_values("Timestamp").reset_index(drop=True)
train, val, test = splits.time_split(tx["Timestamp"])
tx.head()

> Evaluation standard: a naive baseline and a transparent model before any tree; PR-AUC against the base rate, Brier score and a calibration check for classifiers; the threshold chosen on validation data and reported on test data; SHAP or coefficients for the model that drives the decision; a stability check by period, segment or stock.

## 1. EDA (Task 1)

Fraud rate overall and by Transaction_Type, Merchant_Category, Location, amount decile and account-age band.

## 2. Behaviour features

Amount ÷ Historical_Avg_Transaction, log amount, hour and weekday, new-account flag, and per-customer and per-device counts computed **only from earlier transactions**.

## 3. Supervised model (Task 2)

Logistic regression against random forest and XGBoost with class weights. PR-AUC against the 2.3% base rate, recall at fixed alert volumes, precision in the top 1% of scores.

## 4. Isolation Forest (Task 3)

Fit on features only, contamination about 2.3%. Compare with the supervised ranking: PR-AUC, overlap of the top-scored transactions, frauds each finds that the other misses.

## 5. Hybrid

A weighted blend of the two scores (S9). Keep it only if it beats the supervised model on validation data.

## 6. Cost threshold (Task 4)

Expected cost = ₹10,000 × missed frauds + ₹500 × false alarms (`evaluate.cost_curve`, `evaluate.best_threshold`). Sweep on validation, report on test. Check: with a calibrated model the optimum is near 500 ÷ 10,500 ≈ 5%.

## Handoff to Module 9

Annual cost saved against flagging nothing and flagging all, and alerts per day for the operations team.

In [ ]:
# lines = [handoff.make_line(
#     business_line="...", capacity_cr=..., expected_return_pct=..., expected_loss_pct=...,
#     capital_charge_pct=..., stress_loss_pct=..., ml_edge="validated | weak | none", evidence="...",
#     key_risks=[("risk", "control"), ("risk", "control")],
# )]
# handoff.write_handoff("03", lines)

## Traps to check before handing over

Customer or device history built from the whole file leaks the future. SMOTE before the split leaks synthetic frauds into the test set (TA6).